# Decoding Customer Value — A SQL-Driven Retention Strategy

D2C fashion brand, 3,900 customers. Python for cleaning and feature engineering,
SQL for segmentation, Power BI for the founder dashboard.

**Headline finding:** the promotional programme in this dataset carries no
behavioural signal — discount status is perfectly confounded with gender, and
discounted vs full-price customers are identical on every value measure. The
segmentation is therefore built on purchase behaviour alone.

**Pipeline:** audit -> clean -> engineer features -> segment -> SQL -> export for Power BI

In [3]:
import pandas as pd
import numpy as np
import sqlite3
from scipy import stats

pd.set_option("display.width", 160)

df = pd.read_csv(r"D:\downloads\Dataset.csv")         
df.columns = df.columns.str.strip().str.replace("\ufeff", "", regex=False)

print("Shape:", df.shape)
df.head()

Shape: (3900, 18)


,Customer ID,Age,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,Review Rating,Subscription Status,Shipping Type,Discount Applied,Promo Code Used,Previous Purchases,Payment Method,Frequency of Purchases
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,Yes,14,Venmo,Fortnightly
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,Yes,2,Cash,Fortnightly
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,Yes,23,Credit Card,Weekly
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,Yes,49,PayPal,Weekly
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,Yes,31,PayPal,Annually


In [18]:
print("Nulls:")
print(df.isnull().sum()[lambda s: s > 0])
print("\nDuplicate customer IDs:", df["Customer ID"].duplicated().sum())
print("Fully duplicated rows   :", df.duplicated().sum())

print("\nCategorical levels:")
for c in df.columns:
    if pd.api.types.is_numeric_dtype(df[c]):
        continue
    u = df[c].dropna().unique()
    print(f"  {c:24} {len(u):3} -> {list(u) if len(u) <= 8 else '(too many to list)'}")

Nulls:
Review Rating    37
dtype: int64

Duplicate customer IDs: 0
Fully duplicated rows   : 0

Categorical levels:
  Gender                     2 -> ['Male', 'Female']
  Item Purchased            25 -> (too many to list)
  Category                   4 -> ['Clothing', 'Footwear', 'Outerwear', 'Accessories']
  Location                  50 -> (too many to list)
  Size                       4 -> ['L', 'S', 'M', 'XL']
  Color                     25 -> (too many to list)
  Season                     4 -> ['Winter', 'Spring', 'Summer', 'Fall']
  Subscription Status        2 -> ['Yes', 'No']
  Shipping Type              6 -> ['Express', 'Free Shipping', 'Next Day Air', 'Standard', '2-Day Shipping', 'Store Pickup']
  Discount Applied           2 -> ['Yes', 'No']
  Promo Code Used            2 -> ['Yes', 'No']
  Payment Method             6 -> ['Venmo', 'Cash', 'Credit Card', 'PayPal', 'Bank Transfer', 'Debit Card']
  Frequency of Purchases     7 -> ['Fortnightly', 'Weekly', 'Annually', 'Quarte

## 1. Data audit

Nothing is modified here. Every cleaning decision in section 2 has to be
justified by something printed in this section.

### 1a. Column redundancy and the gender confound

Cross-tabs. The **zeros** are what matter — a zero means that combination never
occurs in 3,900 rows.

In [19]:
print("Discount Applied x Promo Code Used")
print(pd.crosstab(df["Discount Applied"], df["Promo Code Used"]))
print("Disagreements:", (df["Discount Applied"] != df["Promo Code Used"]).sum())

print("\n\nGender x Discount Applied")
print(pd.crosstab(df["Gender"], df["Discount Applied"], margins=True))

print("\n\nSubscription Status x Discount Applied")
print(pd.crosstab(df["Subscription Status"], df["Discount Applied"], margins=True))

Discount Applied x Promo Code Used
Promo Code Used     No   Yes
Discount Applied            
No                2223     0
Yes                  0  1677
Disagreements: 0


Gender x Discount Applied
Discount Applied    No   Yes   All
Gender                            
Female            1248     0  1248
Male               975  1677  2652
All               2223  1677  3900


Subscription Status x Discount Applied
Discount Applied       No   Yes   All
Subscription Status                  
No                   2223   624  2847
Yes                     0  1053  1053
All                  2223  1677  3900


In [7]:
# Is the gender/discount pattern explainable by chance?
ct = pd.crosstab(df["Gender"], df["Discount Applied"])
chi2, p, dof, expected = stats.chi2_contingency(ct)

print("Observed:")
print(ct)
print("\nExpected if gender and discount were independent:")
print(pd.DataFrame(expected, index=ct.index, columns=ct.columns).round(1))
print(f"\nchi2 = {chi2:.1f}   p = {p:.3g}")

Observed:
Discount Applied    No   Yes
Gender                      
Female            1248     0
Male               975  1677

Expected if gender and discount were independent:
Discount Applied      No     Yes
Gender                          
Female             711.4   536.6
Male              1511.6  1140.4

chi2 = 1381.9   p = 1.76e-302


**Audit findings**

| # | Finding | Consequence |
|---|---|---|
| 1 | `Discount Applied` and `Promo Code Used` agree on 100% of rows | Drop one — the same signal would otherwise enter twice |
| 2 | Zero of 1,248 female customers ever received a discount (expected ~537 under independence, p ≈ 1.8e-302) | "Promo-dependent" and "male" identify the same people. Any promo-based segment is a gender segment. |
| 3 | Zero subscribers paid full price — subscription is a strict subset of discount | `Subscription Status` adds no information |
| 4 | 37 nulls in `Review Rating`, all male | Not missing at random — mean imputation would bias the exact group the bias came from |

Finding 2 is the one that shapes the rest of the project: the promo axis is
dropped from the segmentation and reported as an artifact instead.

## 2. Cleaning

Four decisions, each traceable to the audit above.

In [8]:
clean = df.copy()

# (1) Drop the redundant column
clean = clean.drop(columns=["Promo Code Used"])

# (2) Merge duplicate cadence labels: 7 labels describe 5 distinct intervals.
#     "Fortnightly" == "Bi-Weekly"; "Every 3 Months" == "Quarterly".
#     Tested with Welch t-tests on basket, history and age: 1 of 6 comparisons
#     flagged at raw p=0.030, which does not survive Bonferroni correction
#     (alpha=0.0083) and carries a negligible effect size (d=-0.128).
freq_map = {"Weekly": "Weekly",
            "Bi-Weekly": "Bi-Weekly", "Fortnightly": "Bi-Weekly",
            "Monthly": "Monthly",
            "Quarterly": "Quarterly", "Every 3 Months": "Quarterly",
            "Annually": "Annually"}
clean["cadence"] = clean["Frequency of Purchases"].map(freq_map)

# (3) Review Rating nulls kept as NaN — see audit finding 4, not missing at random
# (4) No duplicate rows to drop

print("Columns:", clean.shape[1], "(dropped 1, added 1)")
print(clean["cadence"].value_counts())

Columns: 18 (dropped 1, added 1)
cadence
Quarterly    1147
Bi-Weekly    1089
Annually      572
Monthly       553
Weekly        539
Name: count, dtype: int64


## 3. Feature engineering

Each feature answers a question the brand would act on. The dataset has one row
per customer and no timestamps, so tenure and value are proxies — noted where
that matters.

In [9]:
# purchases_per_year — turns a text cadence into a number you can rank and multiply
cadence_map = {"Weekly": 52, "Bi-Weekly": 26, "Monthly": 12, "Quarterly": 4, "Annually": 1}
clean["purchases_per_year"] = clean["cadence"].map(cadence_map)

# forward_annual_value — what this customer is worth over the NEXT 12 months.
# Retention budget should follow future worth, not past worth.
clean["forward_annual_value"] = clean["purchases_per_year"] * clean["Purchase Amount (USD)"]

# satisfaction_flag — Unknown kept as its own level, never imputed.
# NOTE: top bin edge is 5.01 so that perfect 5.0 ratings are included.
clean["satisfaction_flag"] = pd.cut(
    clean["Review Rating"], bins=[0, 3.0, 4.0, 5.01],
    labels=["Low", "Medium", "High"], right=False
).cat.add_categories("Unknown").fillna("Unknown")

# value_tier — quartiles of forward value, for the customer pyramid panel
clean["value_tier"] = pd.qcut(clean["forward_annual_value"], 4,
                              labels=["Tier 4", "Tier 3", "Tier 2", "Tier 1"])

assert (clean["satisfaction_flag"] == "Unknown").sum() == clean["Review Rating"].isna().sum()

print(clean["satisfaction_flag"].value_counts(), "\n")
print(clean.groupby("cadence", observed=True)["forward_annual_value"]
        .agg(["count", "median", "sum"]).sort_values("median", ascending=False))

satisfaction_flag
High       1634
Medium     1549
Low         680
Unknown      37
Name: count, dtype: int64 

           count  median      sum
cadence                          
Weekly       539  3016.0  1652872
Bi-Weekly   1089  1586.0  1695382
Monthly      553   696.0   393720
Quarterly   1147   244.0   275436
Annually     572    59.0    34419


**Value is driven by frequency, not basket size.** Median forward value runs
from \$3,016/yr (Weekly) to \$59/yr (Annually) — a 51x spread — while average
basket size is flat at roughly \$60 across every cadence group. Weekly and
Bi-Weekly buyers are 42% of customers and 83% of forward revenue.

## 4. Segmentation

The dataset has no loyalty column, so loyalty is constructed. Two competing
definitions were built and tested; both initial versions failed and were rebuilt:

- **Definition A** (cadence + history + full price) drew 53.7% women against a
  32.0% baseline — the full-price filter is a gender filter, per audit finding 2.
- **Definition B** (top value quartile + satisfaction) showed identical
  concentration with and without the satisfaction filter (2.60 either way), and
  its customers matched the population average on every loyalty measure.

**A2** — cadence + purchase history, no discount field — is the surviving
definition. It uses no value information yet still captures 48.4% of forward
revenue from 28.9% of customers (1.67x concentration), and its gender mix
(29.9%) sits at baseline.

In [10]:
# A2: behavioural loyalty. Deliberately excludes discount status (= gender).
clean["loyal_A2"] = ((clean["purchases_per_year"] >= 12) &
                     (clean["Previous Purchases"] >= 25)).astype(int)

# B3: value-based, nests strictly inside A2 -> used as the top tier, not a rival
clean["loyal_B3"] = ((clean["forward_annual_value"] >= clean["forward_annual_value"].quantile(0.75)) &
                     (clean["Previous Purchases"] >= clean["Previous Purchases"].median())).astype(int)


def assign_segment(r):
    if r["loyal_B3"] == 1:
        return "1. Core Loyal"
    if r["loyal_A2"] == 1:
        return "2. Established"
    if r["purchases_per_year"] >= 12 or r["Previous Purchases"] >= 25:
        return "3. Developing"
    return "4. Low Engagement"


clean["segment"] = clean.apply(assign_segment, axis=1)

rev_total = clean["forward_annual_value"].sum()
summary = clean.groupby("segment").agg(
    customers=("Customer ID", "count"),
    revenue=("forward_annual_value", "sum"),
    avg_value=("forward_annual_value", "mean"),
    avg_history=("Previous Purchases", "mean"),
    avg_orders_yr=("purchases_per_year", "mean"),
    pct_female=("Gender", lambda g: (g == "Female").mean() * 100),
    pct_discounted=("Discount Applied", lambda d: (d == "Yes").mean() * 100),
)
summary["pct_customers"] = summary["customers"] / len(clean) * 100
summary["pct_revenue"] = summary["revenue"] / rev_total * 100
print(summary[["customers", "pct_customers", "pct_revenue", "avg_value",
               "avg_history", "avg_orders_yr", "pct_female", "pct_discounted"]].round(1))
print(f"\nbaseline: female {(clean['Gender']=='Female').mean()*100:.1f}%, "
      f"discounted {(clean['Discount Applied']=='Yes').mean()*100:.1f}%")

                   customers  pct_customers  pct_revenue  avg_value  avg_history  avg_orders_yr  pct_female  pct_discounted
segment                                                                                                                    
1. Core Loyal            522           13.4         34.8     2703.2         37.4           38.4        29.1            43.9
2. Established           606           15.5         13.5      905.1         37.0           21.2        30.5            44.9
3. Developing           1939           49.7         48.1     1004.1         24.0           16.9        33.7            41.8
4. Low Engagement        833           21.4          3.6      174.5         12.4            2.9        31.0            43.9

baseline: female 32.0%, discounted 43.0%


`pct_female` (29–34%) and `pct_discounted` (42–45%) sit at baseline in every
segment — confirming the scheme separates customers on behaviour, not on the
gender artifact.

## 5. SQL layer

Loaded into SQLite as a fact table plus a region dimension, so the queries below
use joins, CTEs and window functions rather than flat aggregates.

In [11]:
final = clean[[
    "Customer ID", "Age", "Gender", "Category", "Purchase Amount (USD)", "Location",
    "Season", "Review Rating", "Shipping Type", "Discount Applied", "Previous Purchases",
    "Payment Method", "cadence", "purchases_per_year", "forward_annual_value",
    "satisfaction_flag", "value_tier", "segment",
]].rename(columns={
    "Customer ID": "customer_id", "Age": "age", "Gender": "gender", "Category": "category",
    "Purchase Amount (USD)": "purchase_amount", "Location": "location", "Season": "season",
    "Review Rating": "review_rating", "Shipping Type": "shipping_type",
    "Discount Applied": "discount_applied", "Previous Purchases": "previous_purchases",
    "Payment Method": "payment_method",
})
final["value_tier"] = final["value_tier"].astype(str)
final["satisfaction_flag"] = final["satisfaction_flag"].astype(str)

regions = {
    "Northeast": ["Connecticut", "Maine", "Massachusetts", "New Hampshire", "Rhode Island",
                  "Vermont", "New Jersey", "New York", "Pennsylvania"],
    "Midwest": ["Illinois", "Indiana", "Michigan", "Ohio", "Wisconsin", "Iowa", "Kansas",
                "Minnesota", "Missouri", "Nebraska", "North Dakota", "South Dakota"],
    "South": ["Delaware", "Florida", "Georgia", "Maryland", "North Carolina", "South Carolina",
              "Virginia", "West Virginia", "Alabama", "Kentucky", "Mississippi", "Tennessee",
              "Arkansas", "Louisiana", "Oklahoma", "Texas"],
    "West": ["Arizona", "Colorado", "Idaho", "Montana", "Nevada", "New Mexico", "Utah",
             "Wyoming", "Alaska", "California", "Hawaii", "Oregon", "Washington"],
}
dim_region = pd.DataFrame([(s, r) for r, ss in regions.items() for s in ss],
                          columns=["location", "region"])

conn = sqlite3.connect("d2c.db")
final.to_sql("customers", conn, if_exists="replace", index=False)
dim_region.to_sql("dim_region", conn, if_exists="replace", index=False)

unmapped = pd.read_sql("""SELECT COUNT(*) AS unmapped FROM customers c
                          LEFT JOIN dim_region d ON c.location = d.location
                          WHERE d.region IS NULL""", conn)
print(f"customers: {len(final)} rows, {final.shape[1]} cols")
print(f"dim_region: {len(dim_region)} states")
print(f"unmapped states: {unmapped['unmapped'][0]}")

customers: 3900 rows, 18 cols
dim_region: 50 states
unmapped states: 0


### Q1. Who is genuinely loyal vs discount-driven?

In [12]:
q1 = """
SELECT
    segment,
    COUNT(*)                                              AS customers,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1)    AS pct_customers,
    ROUND(AVG(forward_annual_value), 0)                   AS avg_annual_value,
    ROUND(100.0 * SUM(forward_annual_value)
          / SUM(SUM(forward_annual_value)) OVER (), 1)    AS pct_revenue,
    ROUND(AVG(previous_purchases), 1)                     AS avg_history,
    ROUND(100.0 * SUM(CASE WHEN discount_applied = 'Yes' THEN 1 ELSE 0 END)
          / COUNT(*), 1)                                  AS pct_discounted
FROM customers
GROUP BY segment
ORDER BY segment;
"""
pd.read_sql(q1, conn)

,segment,customers,pct_customers,avg_annual_value,pct_revenue,avg_history,pct_discounted
0,1. Core Loyal,522,13.4,2703.0,34.8,37.4,43.9
1,2. Established,606,15.5,905.0,13.5,37.0,44.9
2,3. Developing,1939,49.7,1004.0,48.1,24.0,41.8
3,4. Low Engagement,833,21.4,175.0,3.6,12.4,43.9


`pct_discounted` is flat at 42–45% across all four segments (baseline 43.0%).
Discount status does not distinguish loyal customers from disengaged ones — the
question the brand wanted answered cannot be answered from this field.

### Q2. What behaviour today predicts high value?

In [13]:
q2 = """
SELECT
    cadence,
    COUNT(*)                                           AS customers,
    ROUND(AVG(purchase_amount), 2)                     AS avg_basket,
    ROUND(AVG(previous_purchases), 1)                  AS avg_history,
    ROUND(AVG(forward_annual_value), 0)                AS avg_annual_value,
    ROUND(100.0 * SUM(forward_annual_value)
          / SUM(SUM(forward_annual_value)) OVER (), 1) AS pct_revenue
FROM customers
GROUP BY cadence
ORDER BY avg_annual_value DESC;
"""
pd.read_sql(q2, conn)

,cadence,customers,avg_basket,avg_history,avg_annual_value,pct_revenue
0,Weekly,539,58.97,25.8,3067.0,40.8
1,Bi-Weekly,1089,59.88,25.0,1557.0,41.8
2,Monthly,553,59.33,25.3,712.0,9.7
3,Quarterly,1147,60.03,25.9,240.0,6.8
4,Annually,572,60.17,24.6,60.0,0.8


Basket size is flat (\$59–60) and history is flat (~25) across every cadence
band, while annual value spans 51x. **Purchase frequency is the only driver of
customer value in this business.**

### Q3. Which geographies are commercially underlevered?

Uses a CTE and a join to the region dimension.

In [14]:
q3 = """
WITH regional AS (
    SELECT d.region,
           c.customer_id,
           c.forward_annual_value,
           CASE WHEN c.segment IN ('1. Core Loyal', '2. Established')
                THEN 1 ELSE 0 END AS is_loyal
    FROM customers c
    JOIN dim_region d ON c.location = d.location
)
SELECT
    region,
    COUNT(*)                                            AS customers,
    ROUND(AVG(forward_annual_value), 0)                 AS avg_annual_value,
    SUM(forward_annual_value)                           AS total_value,
    ROUND(100.0 * SUM(forward_annual_value)
          / SUM(SUM(forward_annual_value)) OVER (), 1)  AS pct_revenue,
    ROUND(100.0 * SUM(is_loyal) / COUNT(*), 1)          AS pct_loyal
FROM regional
GROUP BY region
ORDER BY total_value DESC;
"""
print(pd.read_sql(q3, conn).to_string(index=False))

# Is state-level variation real, or sampling noise?
g = pd.read_sql("SELECT location, forward_annual_value FROM customers", conn)
f_stat, p_val = stats.f_oneway(*[x["forward_annual_value"].values
                                 for _, x in g.groupby("location")])
print(f"\nANOVA across 50 states: F = {f_stat:.3f}, p = {p_val:.3f}")

   region  customers  avg_annual_value  total_value  pct_revenue  pct_loyal
    South       1271            1066.0      1355452         33.5       29.0
     West       1018            1051.0      1070141         26.4       30.3
  Midwest        937            1005.0       941708         23.2       27.9
Northeast        674            1016.0       684528         16.9       28.3

ANOVA across 50 states: F = 1.216, p = 0.145


ANOVA p = 0.145 — state-level differences in customer value are **not
distinguishable from random variation**. There is no underlevered geography to
target here. The map panel therefore reports customer *volume* by region, which
is real, and flags value differences as not statistically significant.

### Q4. Is the promotional programme working?

In [15]:
q4 = """
SELECT
    discount_applied,
    COUNT(*)                             AS customers,
    ROUND(AVG(purchase_amount), 2)       AS avg_basket,
    ROUND(AVG(purchases_per_year), 1)    AS avg_orders_per_year,
    ROUND(AVG(previous_purchases), 1)    AS avg_history,
    ROUND(AVG(forward_annual_value), 0)  AS avg_annual_value,
    ROUND(AVG(review_rating), 2)         AS avg_rating,
    ROUND(100.0 * SUM(CASE WHEN gender = 'Female' THEN 1 ELSE 0 END)
          / COUNT(*), 1)                 AS pct_female
FROM customers
GROUP BY discount_applied;
"""
pd.read_sql(q4, conn)

,discount_applied,customers,avg_basket,avg_orders_per_year,avg_history,avg_annual_value,avg_rating,pct_female
0,No,2223,60.13,17.4,25.1,1042.0,3.76,56.1
1,Yes,1677,59.28,17.6,25.7,1035.0,3.74,0.0


Discounted and full-price customers are identical on basket, cadence, history,
annual value and satisfaction. They differ on exactly one attribute: gender
(56.1% vs 0.0%). **The promotional programme segments customers by gender, not
by price sensitivity, and produces no measurable behavioural difference.**

### Q5. What does the ideal customer look like?

In [16]:
q5 = """
SELECT
    CASE WHEN segment IN ('1. Core Loyal', '2. Established')
         THEN 'Loyal' ELSE 'Everyone else' END AS grp,
    COUNT(*)                             AS customers,
    ROUND(AVG(age), 1)                   AS avg_age,
    ROUND(AVG(purchase_amount), 2)       AS avg_basket,
    ROUND(AVG(purchases_per_year), 1)    AS avg_orders_per_year,
    ROUND(AVG(previous_purchases), 1)    AS avg_history,
    ROUND(AVG(forward_annual_value), 0)  AS avg_annual_value,
    ROUND(AVG(review_rating), 2)         AS avg_rating,
    ROUND(100.0 * SUM(CASE WHEN gender = 'Female' THEN 1 ELSE 0 END)
          / COUNT(*), 1)                 AS pct_female
FROM customers
GROUP BY grp;
"""
print(pd.read_sql(q5, conn).to_string(index=False))

# category mix, loyal vs overall
q5b = """
SELECT category,
       ROUND(100.0 * SUM(CASE WHEN segment IN ('1. Core Loyal','2. Established')
                              THEN 1 ELSE 0 END)
             / SUM(SUM(CASE WHEN segment IN ('1. Core Loyal','2. Established')
                            THEN 1 ELSE 0 END)) OVER (), 1) AS pct_of_loyal,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1)   AS pct_of_all
FROM customers
GROUP BY category
ORDER BY pct_of_all DESC;
"""
print()
print(pd.read_sql(q5b, conn).to_string(index=False))

          grp  customers  avg_age  avg_basket  avg_orders_per_year  avg_history  avg_annual_value  avg_rating  pct_female
Everyone else       2772     43.8       59.78                 12.7         20.5             755.0        3.75        32.9
        Loyal       1128     44.8       59.72                 29.1         37.2            1737.0        3.76        29.9

   category  pct_of_loyal  pct_of_all
   Clothing          44.5        44.5
Accessories          31.4        31.8
   Footwear          15.4        15.4
  Outerwear           8.7         8.3


**Ideal customer profile:** buys monthly or more often, with 25+ prior
purchases. Loyal customers are *not* distinguishable from the average customer by
age (44.8 vs 43.8), basket size (\$59.72 vs \$59.78, p = 0.94), category mix, or
satisfaction. Two attributes separate them, and only two: orders per year
(29.1 vs 12.7) and purchase history (37.2 vs 20.5).

The implication for marketing is to stop building demographic personas and
optimise for repeat rate instead.

## 6. Export for Power BI

In [17]:
final.to_csv("customers_powerbi.csv", index=False)
dim_region.to_csv("dim_region.csv", index=False)
conn.close()

print("customers_powerbi.csv :", final.shape)
print("dim_region.csv        :", dim_region.shape)

customers_powerbi.csv : (3900, 18)
dim_region.csv        : (50, 2)


### Dashboard panels

| Panel | Build | Note |
|---|---|---|
| Customer pyramid | Stacked bar — `value_tier` by customer count and summed `forward_annual_value` | Tier 1 = 25% of customers, 65% of revenue |
| Value by cadence | Column chart — `cadence` vs avg `forward_annual_value`, with avg `purchase_amount` as a flat reference line | Shows frequency drives value, basket size does not |
| Regional volume | Filled map on `location`, joined to `dim_region` | Label value differences as not statistically significant (ANOVA p = 0.145) |
| Category mix | Clustered bar — `category` share among loyal vs all customers | The two mixes are near-identical, which is the finding |